
# 90-Day Daily LSTM Input for Insider NMS

This notebook converts the existing `model_windows`, `trx_vw`, and `df_acct` logic into a fixed-length daily sequence for an LSTM.

## Design

Each model sample is:

- one `login_id`
- one `acct_nbr`
- one `anchor_date`
- 90 calendar days ending on `anchor_date`
- one label

The LSTM input shape is:

```text
number_of_samples × 90 days × number_of_daily_features
```

The daily features are inspired by the original 20 LightGBM features, but are represented in a time-series-friendly way.

Important leakage rule:

```text
sequence_start = anchor_date - 89 days
sequence_end   = anchor_date
```

We do not use transactions after `anchor_date`, even when `fraud_date` is later.


In [ ]:

# Databricks notebook setup

%run ../utils/load_tables_utils

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import ArrayType, DoubleType

import numpy as np
import pandas as pd


In [ ]:

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MODEL_WINDOWS_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_windows_v2"
)

TRX_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_trx_alert_table"
)

OUTPUT_LONG_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_lstm_daily_long_v1"
)

OUTPUT_ARRAY_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_lstm_daily_array_v1"
)

LOOKBACK_DAYS = 90


In [ ]:

# ------------------------------------------------------------
# Load the two main tables
# ------------------------------------------------------------

model_windows = spark.read.format("delta").load(MODEL_WINDOWS_PATH)
trx_vw = spark.read.format("delta").load(TRX_PATH)

print("model_windows rows:", model_windows.count())
print("trx_vw rows:", trx_vw.count())

model_windows.printSchema()
trx_vw.printSchema()



## 1. Build the sample spine

The sample key is:

```text
login_id + acct_nbr + anchor_date
```

`anchor_date` is the original `model_windows.date`.

We keep the original label and split information, but create a leakage-safe 90-day input ending on the anchor date.


In [ ]:

# Keep only the columns needed for the LSTM sample definition.

sample_spine = (
    model_windows
    .select(
        "login_id",
        "acct_nbr",
        F.to_date("date").alias("anchor_date"),
        "label",
        "insider_label",
        "label_split",
        "cv_fold",
        "fraud_date",
        "lookback_window_start",
        "total_related_seq_dates"
    )
)

# Check whether the same sample key has conflicting labels.
sample_conflicts = (
    sample_spine
    .groupBy("login_id", "acct_nbr", "anchor_date")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("label").alias("distinct_label_count"),
        F.countDistinct("label_split").alias("distinct_split_count")
    )
    .filter(
        (F.col("distinct_label_count") > 1)
        | (F.col("distinct_split_count") > 1)
    )
)

if sample_conflicts.limit(1).count() > 0:
    display(sample_conflicts)
    raise ValueError(
        "Conflicting labels or splits exist for the same "
        "login_id + acct_nbr + anchor_date."
    )

sample_spine = (
    sample_spine
    .dropDuplicates(["login_id", "acct_nbr", "anchor_date"])
    .withColumn(
        "sequence_start",
        F.date_sub("anchor_date", LOOKBACK_DAYS - 1)
    )
)



## 2. Create a complete 90-day calendar

Each sample must have exactly 90 rows.

Inactive dates are retained because the absence of employee-account activity is informative for an LSTM.


In [ ]:

daily_spine = (
    sample_spine
    .withColumn(
        "calendar_date",
        F.explode(
            F.sequence(
                F.col("sequence_start"),
                F.col("anchor_date"),
                F.expr("interval 1 day")
            )
        )
    )
    .withColumn(
        "day_index",
        F.datediff("calendar_date", "sequence_start")
    )
    .withColumn(
        "is_related_seq_date",
        F.array_contains(
            F.col("total_related_seq_dates"),
            F.col("calendar_date")
        ).cast("int")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("calendar_date").isin([1, 7]).cast("int")
    )
)

# Every sample should have exactly 90 calendar rows.
daily_count_check = (
    daily_spine
    .groupBy("login_id", "acct_nbr", "anchor_date")
    .count()
    .groupBy("count")
    .count()
)

display(daily_count_check)



## 3. Transaction-level preparation

This follows the original feature-generation logic:

- transaction type `3` is inquiry
- transaction type `4` is maintenance
- after-hours depends on the original weekday-specific lookup
- transaction gaps are calculated within the same employee-account-day


In [ ]:

# Original weekday-specific business hours from feature_generation.ipynb.

after_hours_lookup = spark.createDataFrame(
    [
        (1, 9, 30, 15, 30),
        (2, 7, 0, 18, 30),
        (3, 7, 0, 18, 30),
        (4, 7, 0, 18, 30),
        (5, 7, 0, 19, 30),
        (6, 7, 0, 19, 30),
        (7, 7, 30, 14, 30),
    ],
    [
        "transaction_day",
        "start_hour",
        "start_minute",
        "end_hour",
        "end_minute",
    ],
)

trx_base = (
    trx_vw
    .withColumn("calendar_date", F.to_date("date"))
    .withColumn(
        "is_inquiry",
        (F.col("transaction_type").cast("string") == "3").cast("int")
    )
    .withColumn(
        "is_maintenance",
        (F.col("transaction_type").cast("string") == "4").cast("int")
    )
    .join(after_hours_lookup, on="transaction_day", how="left")
    .withColumn(
        "is_after_hours",
        F.when(
            (
                (F.col("transaction_hour") < F.col("start_hour"))
                | (
                    (F.col("transaction_hour") == F.col("start_hour"))
                    & (
                        F.col("transaction_minute")
                        < F.col("start_minute")
                    )
                )
            )
            | (
                (F.col("transaction_hour") > F.col("end_hour"))
                | (
                    (F.col("transaction_hour") == F.col("end_hour"))
                    & (
                        F.col("transaction_minute")
                        > F.col("end_minute")
                    )
                )
            ),
            1,
        ).otherwise(0),
    )
)

# Calculate transaction gaps only within the same day.
daily_gap_window = (
    Window
    .partitionBy("login_id", "acct_nbr", "calendar_date")
    .orderBy("transaction_datetime")
)

trx_base = (
    trx_base
    .withColumn(
        "previous_transaction_datetime",
        F.lag("transaction_datetime").over(daily_gap_window)
    )
    .withColumn(
        "gap_minutes",
        (
            F.col("transaction_datetime").cast("long")
            - F.col("previous_transaction_datetime").cast("long")
        ) / F.lit(60.0)
    )
)


In [ ]:

# Validate transaction-type mapping before relying on the indicators.
display(
    trx_vw
    .groupBy(
        "transaction_type",
        "internal_transaction_description"
    )
    .count()
    .orderBy(F.desc("count"))
)



## 4. Daily transaction features

These are daily versions of the original touch features.

Original window feature | Daily LSTM feature
---|---
`inquiries` | `daily_inquiries`
`maintenances` | `daily_maintenances`
`prop_inquiries` | `daily_prop_inquiries`
`prop_maintenances` | `daily_prop_maintenances`
`after_hours_touches` | `daily_after_hours_touches`
`avg_time_bw_touch_dt` | `daily_avg_gap_minutes`


In [ ]:

transaction_daily = (
    trx_base
    .groupBy("login_id", "acct_nbr", "calendar_date")
    .agg(
        F.count("*").alias("daily_touches"),
        F.sum("is_inquiry").alias("daily_inquiries"),
        F.sum("is_maintenance").alias("daily_maintenances"),
        F.sum("is_after_hours").alias(
            "daily_after_hours_touches"
        ),
        F.sum(F.coalesce(F.col("is_non_mon"), F.lit(0))).alias(
            "daily_non_mon_touches"
        ),
        F.sum(F.coalesce(F.col("is_mon"), F.lit(0))).alias(
            "daily_mon_touches"
        ),
        F.avg("gap_minutes").alias("daily_avg_gap_minutes"),
        F.max("gap_minutes").alias("daily_max_gap_minutes"),
        F.countDistinct("channel_id_cd").alias(
            "daily_distinct_channels"
        ),
    )
    .withColumn(
        "daily_prop_inquiries",
        F.when(
            F.col("daily_touches") > 0,
            F.col("daily_inquiries") / F.col("daily_touches")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "daily_prop_maintenances",
        F.when(
            F.col("daily_touches") > 0,
            F.col("daily_maintenances") / F.col("daily_touches")
        ).otherwise(F.lit(0.0))
    )
)



## 5. Daily sequence features

Sequence-level fields are repeated on every transaction row. Therefore, the code first deduplicates each sequence before calculating daily sequence statistics.

The composite sequence key uses:

```text
login_id
acct_nbr
date
min_seq_datetime
max_seq_datetime
seq_len
```


In [ ]:

sequence_level = (
    trx_vw
    .select(
        "login_id",
        "acct_nbr",
        F.to_date("date").alias("calendar_date"),
        "min_seq_datetime",
        "max_seq_datetime",
        "seq_len",
        "seq_len_in_time",
        "non_mon_seq_ind",
        "seq_non_mon_touches",
        "seq_mon_touches",
    )
    .dropDuplicates(
        [
            "login_id",
            "acct_nbr",
            "calendar_date",
            "min_seq_datetime",
            "max_seq_datetime",
            "seq_len",
        ]
    )
)

sequence_daily = (
    sequence_level
    .groupBy("login_id", "acct_nbr", "calendar_date")
    .agg(
        F.count("*").alias("daily_sequence_count"),
        F.sum(
            F.coalesce(F.col("non_mon_seq_ind"), F.lit(0))
        ).alias("daily_non_mon_sequence_count"),
        F.avg("seq_len").alias(
            "daily_mean_sequence_touch_count"
        ),
        F.max("seq_len").alias(
            "daily_max_sequence_touch_count"
        ),
        F.avg("seq_len_in_time").alias(
            "daily_mean_sequence_duration_seconds"
        ),
        F.max("seq_len_in_time").alias(
            "daily_max_sequence_duration_seconds"
        ),
    )
)



## 6. Prepare daily account history

This follows the original account-history cleaning logic:

- collapse adjacent intervals when balance and normal-status do not change
- remove overlapping end dates
- map each calendar date to its active account-history interval


In [ ]:

normal_acct_statuses = ["active", "normal"]

acct_base_window = (
    Window
    .partitionBy("acct_nbr")
    .orderBy("effective_date")
)

acct_attrs = (
    df_acct
    .withColumn(
        "is_normal",
        F.when(
            F.lower(F.col("acct_status_cd_description"))
            .isin(normal_acct_statuses),
            1
        ).otherwise(0)
    )
    .select(
        "effective_date",
        "effective_end_date",
        "acct_nbr",
        "acct_balance",
        "is_normal",
    )
    .withColumn(
        "prev_effective_end_date",
        F.lag("effective_end_date").over(acct_base_window)
    )
    .withColumn(
        "prev_acct_balance",
        F.lag("acct_balance").over(acct_base_window)
    )
    .withColumn(
        "prev_is_normal",
        F.lag("is_normal").over(acct_base_window)
    )
    .withColumn(
        "is_new_group",
        F.when(
            F.col("prev_effective_end_date").isNull()
            | (
                F.datediff(
                    "effective_date",
                    "prev_effective_end_date"
                ) > 1
            )
            | (
                F.col("acct_balance")
                != F.col("prev_acct_balance")
            )
            | (
                F.col("is_normal")
                != F.col("prev_is_normal")
            ),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "grp",
        F.sum("is_new_group").over(
            acct_base_window.rowsBetween(
                Window.unboundedPreceding,
                Window.currentRow,
            )
        ),
    )
    .groupBy(
        "acct_nbr",
        "acct_balance",
        "is_normal",
        "grp",
    )
    .agg(
        F.min("effective_date").alias("start_date"),
        F.max("effective_end_date").alias("end_date"),
    )
    .withColumn(
        "next_start_date",
        F.lead("start_date").over(
            Window
            .partitionBy("acct_nbr")
            .orderBy("start_date")
        ),
    )
    .withColumn(
        "end_date",
        F.when(
            F.col("next_start_date") == F.col("end_date"),
            F.date_sub("end_date", 1),
        ).otherwise(F.col("end_date")),
    )
    # Protect against invalid single-day intervals after adjustment.
    .filter(F.col("end_date") >= F.col("start_date"))
    .select(
        "acct_nbr",
        "start_date",
        "end_date",
        "acct_balance",
        "is_normal",
    )
)



## 7. Join all daily information to the 90-day spine


In [ ]:

daily_features = (
    daily_spine.alias("s")
    .join(
        transaction_daily.alias("t"),
        on=[
            F.col("s.login_id") == F.col("t.login_id"),
            F.col("s.acct_nbr") == F.col("t.acct_nbr"),
            F.col("s.calendar_date") == F.col("t.calendar_date"),
        ],
        how="left",
    )
    .drop(
        F.col("t.login_id"),
        F.col("t.acct_nbr"),
        F.col("t.calendar_date"),
    )
    .join(
        sequence_daily.alias("q"),
        on=[
            F.col("s.login_id") == F.col("q.login_id"),
            F.col("s.acct_nbr") == F.col("q.acct_nbr"),
            F.col("s.calendar_date") == F.col("q.calendar_date"),
        ],
        how="left",
    )
    .drop(
        F.col("q.login_id"),
        F.col("q.acct_nbr"),
        F.col("q.calendar_date"),
    )
)

# Join account state by date-range overlap.
daily_features = (
    daily_features.alias("d")
    .join(
        acct_attrs.alias("a"),
        on=[
            F.col("d.acct_nbr") == F.col("a.acct_nbr"),
            F.col("d.calendar_date").between(
                F.col("a.start_date"),
                F.col("a.end_date"),
            ),
        ],
        how="left",
    )
    .drop(F.col("a.acct_nbr"))
    .select("d.*", "a.acct_balance", "a.is_normal")
)


In [ ]:

# Fill only activity-derived missing values with zero.
# Account balance is handled separately because a missing balance
# is not the same thing as a zero balance.

ZERO_FILL_COLUMNS = [
    "daily_touches",
    "daily_inquiries",
    "daily_maintenances",
    "daily_after_hours_touches",
    "daily_non_mon_touches",
    "daily_mon_touches",
    "daily_avg_gap_minutes",
    "daily_max_gap_minutes",
    "daily_distinct_channels",
    "daily_prop_inquiries",
    "daily_prop_maintenances",
    "daily_sequence_count",
    "daily_non_mon_sequence_count",
    "daily_mean_sequence_touch_count",
    "daily_max_sequence_touch_count",
    "daily_mean_sequence_duration_seconds",
    "daily_max_sequence_duration_seconds",
]

daily_features = (
    daily_features
    .fillna(0, subset=ZERO_FILL_COLUMNS)
    .withColumn(
        "has_touch",
        (F.col("daily_touches") > 0).cast("int")
    )
    .withColumn(
        "has_multiple_touches",
        (F.col("daily_touches") > 1).cast("int")
    )
    .withColumn(
        "acct_balance_missing",
        F.col("acct_balance").isNull().cast("int")
    )
    .withColumn(
        "daily_acct_balance",
        F.coalesce(F.col("acct_balance"), F.lit(0.0))
    )
    .withColumn(
        "daily_is_normal",
        F.coalesce(F.col("is_normal"), F.lit(0))
    )
    .drop("acct_balance", "is_normal")
)



## 8. Create daily change and rolling features

These are daily versions of:

- `change_acct_balance`
- `num_status_changes`
- `stddev_acct_balance`


In [ ]:

sample_day_window = (
    Window
    .partitionBy("login_id", "acct_nbr", "anchor_date")
    .orderBy("day_index")
)

rolling_7d_window = (
    sample_day_window.rowsBetween(-6, 0)
)

rolling_30d_window = (
    sample_day_window.rowsBetween(-29, 0)
)

daily_features = (
    daily_features
    .withColumn(
        "previous_acct_balance",
        F.lag("daily_acct_balance").over(sample_day_window)
    )
    .withColumn(
        "daily_balance_change",
        F.when(
            F.col("day_index") == 0,
            F.lit(0.0)
        ).otherwise(
            F.col("daily_acct_balance")
            - F.col("previous_acct_balance")
        )
    )
    .withColumn(
        "previous_is_normal",
        F.lag("daily_is_normal").over(sample_day_window)
    )
    .withColumn(
        "daily_status_change",
        F.when(
            (F.col("day_index") > 0)
            & (
                F.col("daily_is_normal")
                != F.col("previous_is_normal")
            ),
            1,
        ).otherwise(0),
    )
    .withColumn(
        "balance_std_7d",
        F.coalesce(
            F.stddev_pop("daily_acct_balance").over(
                rolling_7d_window
            ),
            F.lit(0.0),
        ),
    )
    .withColumn(
        "balance_std_30d",
        F.coalesce(
            F.stddev_pop("daily_acct_balance").over(
                rolling_30d_window
            ),
            F.lit(0.0),
        ),
    )
    .drop(
        "previous_acct_balance",
        "previous_is_normal",
    )
)



## 9. Add recency features

`days_since_previous_touch` helps the LSTM distinguish:

- repeated activity
- long inactivity followed by sudden activity
- continuous employee-account access


In [ ]:

previous_rows_window = (
    Window
    .partitionBy("login_id", "acct_nbr", "anchor_date")
    .orderBy("day_index")
    .rowsBetween(Window.unboundedPreceding, -1)
)

daily_features = (
    daily_features
    .withColumn(
        "previous_touch_date",
        F.max(
            F.when(
                F.col("has_touch") == 1,
                F.col("calendar_date")
            )
        ).over(previous_rows_window)
    )
    .withColumn(
        "days_since_previous_touch",
        F.when(
            F.col("previous_touch_date").isNull(),
            F.lit(LOOKBACK_DAYS)
        ).otherwise(
            F.datediff(
                "calendar_date",
                "previous_touch_date"
            )
        )
    )
    .drop("previous_touch_date")
)



## 10. Optional employee-level daily context

These features approximate the original cross-account employee features on a daily basis.

They show whether the current account represents an unusually large share of the employee's activity that day.


In [ ]:

employee_daily_context = (
    transaction_daily
    .groupBy("login_id", "calendar_date")
    .agg(
        F.sum("daily_touches").alias(
            "employee_daily_total_touches"
        ),
        F.countDistinct("acct_nbr").alias(
            "employee_daily_distinct_accounts"
        ),
        F.avg("daily_touches").alias(
            "employee_daily_avg_touches_per_account"
        ),
    )
)

daily_features = (
    daily_features
    .join(
        employee_daily_context,
        on=["login_id", "calendar_date"],
        how="left",
    )
    .fillna(
        0,
        subset=[
            "employee_daily_total_touches",
            "employee_daily_distinct_accounts",
            "employee_daily_avg_touches_per_account",
        ],
    )
    .withColumn(
        "current_account_daily_touch_share",
        F.when(
            F.col("employee_daily_total_touches") > 0,
            F.col("daily_touches")
            / F.col("employee_daily_total_touches"),
        ).otherwise(F.lit(0.0)),
    )
)



## 11. Final daily feature list

This list is intentionally close to the original 20-feature business logic, but is adapted for time-series modeling.


In [ ]:

LSTM_DAILY_FEATURES = [
    # Daily touch volume
    "daily_touches",
    "daily_inquiries",
    "daily_maintenances",
    "daily_after_hours_touches",
    "daily_non_mon_touches",
    "daily_mon_touches",

    # Daily composition
    "daily_prop_inquiries",
    "daily_prop_maintenances",

    # Sequence behavior
    "daily_sequence_count",
    "daily_non_mon_sequence_count",
    "daily_mean_sequence_touch_count",
    "daily_max_sequence_touch_count",
    "daily_mean_sequence_duration_seconds",
    "daily_max_sequence_duration_seconds",

    # Timing behavior
    "daily_avg_gap_minutes",
    "daily_max_gap_minutes",
    "has_multiple_touches",

    # Account behavior
    "daily_acct_balance",
    "daily_balance_change",
    "daily_is_normal",
    "daily_status_change",
    "balance_std_7d",
    "balance_std_30d",
    "acct_balance_missing",

    # Activity pattern
    "has_touch",
    "days_since_previous_touch",
    "is_related_seq_date",
    "is_weekend",

    # Employee-level daily context
    "employee_daily_total_touches",
    "employee_daily_distinct_accounts",
    "employee_daily_avg_touches_per_account",
    "current_account_daily_touch_share",
]

print("Number of daily features:", len(LSTM_DAILY_FEATURES))
print(LSTM_DAILY_FEATURES)



## 12. Create long-format LSTM input

The long table contains one row per sample per day.

This is useful for validation and debugging.


In [ ]:

LSTM_METADATA_COLUMNS = [
    "login_id",
    "acct_nbr",
    "anchor_date",
    "calendar_date",
    "day_index",
    "label",
    "insider_label",
    "label_split",
    "cv_fold",
]

lstm_input_long = (
    daily_features
    .select(
        *LSTM_METADATA_COLUMNS,
        *[
            F.col(c).cast("double").alias(c)
            for c in LSTM_DAILY_FEATURES
        ],
    )
    .orderBy(
        "login_id",
        "acct_nbr",
        "anchor_date",
        "day_index",
    )
)

display(lstm_input_long.limit(200))



## 13. Create one-row-per-sample array input

The resulting `feature_sequence` has shape:

```text
90 × K
```

where `K` is the number of daily features.


In [ ]:

lstm_input_with_vector = (
    lstm_input_long
    .withColumn(
        "daily_feature_vector",
        F.array(
            *[
                F.col(c).cast("double")
                for c in LSTM_DAILY_FEATURES
            ]
        ),
    )
)

lstm_input_array = (
    lstm_input_with_vector
    .groupBy(
        "login_id",
        "acct_nbr",
        "anchor_date",
        "label",
        "insider_label",
        "label_split",
        "cv_fold",
    )
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    "day_index",
                    "daily_feature_vector",
                )
            )
        ).alias("ordered_daily_rows")
    )
    .withColumn(
        "feature_sequence",
        F.transform(
            "ordered_daily_rows",
            lambda x: x["daily_feature_vector"],
        ),
    )
    .withColumn(
        "sequence_length",
        F.size("feature_sequence")
    )
    .drop("ordered_daily_rows")
)

display(lstm_input_array.limit(20))



## 14. Input quality checks


In [ ]:

# Check 1: every sample has exactly 90 days.
invalid_length = lstm_input_array.filter(
    F.col("sequence_length") != LOOKBACK_DAYS
)

print(
    "Samples with invalid sequence length:",
    invalid_length.count()
)

# Check 2: long-table uniqueness.
duplicate_days = (
    lstm_input_long
    .groupBy(
        "login_id",
        "acct_nbr",
        "anchor_date",
        "day_index",
    )
    .count()
    .filter(F.col("count") != 1)
)

print(
    "Duplicate sample-day rows:",
    duplicate_days.count()
)

# Check 3: label and split distribution.
display(
    lstm_input_array
    .groupBy("label_split", "label")
    .count()
    .orderBy("label_split", "label")
)

# Check 4: feature summary.
display(
    lstm_input_long.select(LSTM_DAILY_FEATURES).summary()
)



## 15. Persist the prepared input


In [ ]:

(
    lstm_input_long
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(OUTPUT_LONG_PATH)
)

(
    lstm_input_array
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(OUTPUT_ARRAY_PATH)
)

print("Saved long input to:", OUTPUT_LONG_PATH)
print("Saved array input to:", OUTPUT_ARRAY_PATH)



# Optional: PyTorch prototype

The section below is suitable for a prototype or a sampled dataset.

For a very large dataset, do not call `toPandas()` on the entire table. Use a scalable data loader or export partitioned files.


In [ ]:

# Select a manageable prototype sample.
# Remove or increase the limit only after confirming driver memory.

prototype_spark = (
    lstm_input_array
    .filter(F.col("label_split").isin("train", "val", "test"))
    .select(
        "login_id",
        "acct_nbr",
        "anchor_date",
        "label",
        "label_split",
        "feature_sequence",
    )
    .limit(50000)
)

prototype_pdf = prototype_spark.toPandas()

X = np.stack(
    prototype_pdf["feature_sequence"]
    .apply(lambda x: np.asarray(x, dtype=np.float32))
    .to_numpy()
)

y = prototype_pdf["label"].astype(np.float32).to_numpy()
splits = prototype_pdf["label_split"].to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:

# Fit scaling parameters using TRAIN ONLY to prevent leakage.

train_mask = splits == "train"
val_mask = splits == "val"
test_mask = splits == "test"

feature_mean = X[train_mask].mean(axis=(0, 1), keepdims=True)
feature_std = X[train_mask].std(axis=(0, 1), keepdims=True)

# Avoid division by zero for constant features.
feature_std = np.where(feature_std < 1e-8, 1.0, feature_std)

X_scaled = (X - feature_mean) / feature_std

X_train = X_scaled[train_mask]
y_train = y[train_mask]

X_val = X_scaled[val_mask]
y_val = y[val_mask]

X_test = X_scaled[test_mask]
y_test = y[test_mask]

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


In [ ]:

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 256

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


In [ ]:

class InsiderFraudLSTM(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size=64,
        num_layers=1,
        dropout=0.20,
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        _, (hidden_state, _) = self.lstm(x)

        # Last recurrent layer's final hidden state.
        sequence_embedding = hidden_state[-1]

        logits = self.classifier(sequence_embedding).squeeze(1)
        return logits


model = InsiderFraudLSTM(
    input_size=len(LSTM_DAILY_FEATURES),
    hidden_size=64,
    num_layers=1,
    dropout=0.20,
).to(DEVICE)

print(model)


In [ ]:

# Handle class imbalance using positive-class weighting.

negative_count = float((y_train == 0).sum())
positive_count = float((y_train == 1).sum())

if positive_count == 0:
    raise ValueError("No positive samples exist in the training subset.")

pos_weight_value = negative_count / positive_count

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        pos_weight_value,
        dtype=torch.float32,
        device=DEVICE,
    )
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

print("Positive-class weight:", pos_weight_value)


In [ ]:

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


def evaluate_model(model, data_loader):
    model.eval()

    all_labels = []
    all_probabilities = []

    total_loss = 0.0

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            probabilities = torch.sigmoid(logits)

            total_loss += loss.item() * batch_x.size(0)
            all_labels.append(batch_y.cpu().numpy())
            all_probabilities.append(
                probabilities.cpu().numpy()
            )

    labels = np.concatenate(all_labels)
    probabilities = np.concatenate(all_probabilities)

    average_loss = total_loss / len(data_loader.dataset)

    # Metrics require both classes.
    if len(np.unique(labels)) == 2:
        roc_auc = roc_auc_score(labels, probabilities)
        pr_auc = average_precision_score(
            labels,
            probabilities,
        )
    else:
        roc_auc = np.nan
        pr_auc = np.nan

    return {
        "loss": average_loss,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }


In [ ]:

EPOCHS = 10
best_val_pr_auc = -np.inf
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(batch_x)
        loss = criterion(logits, batch_y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )
        optimizer.step()

        running_loss += loss.item() * batch_x.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_metrics = evaluate_model(model, val_loader)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.6f} | "
        f"val_loss={val_metrics['loss']:.6f} | "
        f"val_roc_auc={val_metrics['roc_auc']:.6f} | "
        f"val_pr_auc={val_metrics['pr_auc']:.6f}"
    )

    current_pr_auc = val_metrics["pr_auc"]

    if not np.isnan(current_pr_auc) and current_pr_auc > best_val_pr_auc:
        best_val_pr_auc = current_pr_auc
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(DEVICE)

test_metrics = evaluate_model(model, test_loader)
print("Final test metrics:", test_metrics)
